In [1]:
from collections import defaultdict
import torch
from datasets import load_dataset, concatenate_datasets, Dataset
from sentence_transformers import (
    SentenceTransformer, 
    SentenceTransformerTrainer, 
    SentenceTransformerTrainingArguments,
    losses
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator

/Users/lucasokamura/miniconda3/envs/probing-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Seleção Automática do Device Ideal
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available(): # Para Mac M1/M2/M3
    device = "mps"
else:
    device = "cpu"
print(f"A utilizar o device: {device}")

A utilizar o device: mps


In [3]:
# 2. Carregamento das 5 partes do Quati 10M
print("A carregar e concatenar as 5 partes do Quati 10M...")
parts = []
for i in range(5):
    config_name = f"quati_10M_passages_part_0{i}"
    print(f" - Lendo {config_name}...")
    part = load_dataset("unicamp-dl/quati", config_name, split=config_name, trust_remote_code=True)
    parts.append(part)

# Concatenar tudo num único objeto de 10 milhões de linhas (mapeado em disco)
passages = concatenate_datasets(parts)
topics = load_dataset("unicamp-dl/quati", "quati_all_topics", split="quati_all_topics", trust_remote_code=True)
qrels = load_dataset("unicamp-dl/quati", "quati_10M_qrels", split="quati_10M_qrels", trust_remote_code=True)

A carregar e concatenar as 5 partes do Quati 10M...
 - Lendo quati_10M_passages_part_00...
 - Lendo quati_10M_passages_part_01...
 - Lendo quati_10M_passages_part_02...
 - Lendo quati_10M_passages_part_03...
 - Lendo quati_10M_passages_part_04...


In [4]:
# 3. Indexação e Mapeamento
print("A indexar passagens...")
docid_to_idx = {str(did): i for i, did in enumerate(passages["passage_id"])}
queryid_to_idx = {str(qid): i for i, qid in enumerate(topics["query_id"])}

A indexar passagens...


In [5]:
# 4. Preparação de Dados para Treino e Eval
query_to_docs = defaultdict(lambda: {"pos": [], "neg": []})
for row in qrels:
    qid = str(row["query_id"])
    if row["score"] == 3: query_to_docs[qid]["pos"].append(str(row["passage_id"]))
    elif row["score"] == 0: query_to_docs[qid]["neg"].append(str(row["passage_id"]))

unique_qids = list(query_to_docs.keys())
train_qids = unique_qids[:int(len(unique_qids) * 0.9)]
eval_qids = unique_qids[int(len(unique_qids) * 0.9):]

def create_triplet_ds(qids):
    anchors, positives, negatives = [], [], []
    for qid in qids:
        if query_to_docs[qid]["pos"] and query_to_docs[qid]["neg"]:
            q_text = topics[queryid_to_idx[qid]]["query"]
            # Selecionamos os pares de forma eficiente
            for p_id in query_to_docs[qid]["pos"][:3]:
                for n_id in query_to_docs[qid]["neg"][:3]:
                    anchors.append(q_text)
                    positives.append(passages[docid_to_idx[p_id]]["passage"])
                    negatives.append(passages[docid_to_idx[n_id]]["passage"])
    return Dataset.from_dict({"anchor": anchors, "positive": positives, "negative": negatives})

train_dataset = create_triplet_ds(train_qids)

In [6]:
train_dataset.to_pandas()

,anchor,positive,negative
0,Qual a maior característica da fauna brasileira?,Bacia do São Francisco. 08. (FGV-SP) No Brasil...,Brasil abriga uma diversidade de ecossistemas ...
1,Qual a maior característica da fauna brasileira?,Bacia do São Francisco. 08. (FGV-SP) No Brasil...,A fauna entomológica cadavérica no Brasil apre...
2,Qual a maior característica da fauna brasileira?,Bacia do São Francisco. 08. (FGV-SP) No Brasil...,19. (FATEC) “Apresenta vegetação arbórea espar...
3,Qual a maior característica da fauna brasileira?,(FGV) No Brasil há a presença de variados biom...,Brasil abriga uma diversidade de ecossistemas ...
4,Qual a maior característica da fauna brasileira?,(FGV) No Brasil há a presença de variados biom...,A fauna entomológica cadavérica no Brasil apre...
...,...,...,...
343,Quais as origens de pessoas com olhos verdes?,"essa etnia forçada a migrar para o Brasil, uma...","Segundo Ana, o planeta Terra produz diariament..."
344,Quais as origens de pessoas com olhos verdes?,"essa etnia forçada a migrar para o Brasil, uma...",A margarida tem folhas verdes que formam uma r...
345,Quais as origens de pessoas com olhos verdes?,Os olhos cor de âmbar são uns dos mais fascina...,Verde – Se quer alguém para conversar e te ouv...
346,Quais as origens de pessoas com olhos verdes?,Os olhos cor de âmbar são uns dos mais fascina...,"Segundo Ana, o planeta Terra produz diariament..."


In [7]:
# 5. Configuração do Avaliador (Sub-amostra para performance)
ir_evaluator = InformationRetrievalEvaluator(
    queries={qid: topics[queryid_to_idx[qid]]["query"] for qid in eval_qids[:100]},
    corpus={str(passages[i]["passage_id"]): passages[i]["passage"] for i in range(5000)}, # Mini-corpus de teste
    relevant_docs={qid: set(query_to_docs[qid]["pos"]) for qid in eval_qids[:100]},
    name="quati_eval"
)

In [ ]:
# 6. Modelo e Treino
model_id = "rufimelo/Legal-BERTimbau-sts-large-ma-v3"
model = SentenceTransformer(model_id, device=device)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

args = SentenceTransformerTrainingArguments(
    output_dir="./legal-quati-10M",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True if device == "cuda" else False, # fp16 apenas para NVIDIA
    eval_strategy="steps",
    eval_steps=200,
    save_steps=600,
    logging_steps=50,
    save_total_limit=2,
    gradient_accumulation_steps=4,
    load_best_model_at_end=True
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=ir_evaluator
)

trainer.train()
model.save_pretrained("./legal-quati-10M-final")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9896.52it/s]
BertModel LOAD REPORT from: rufimelo/Legal-BERTimbau-sts-large-ma-v3
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Invalid model-index. Not loading eval results into CardData.
The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.
/Users/lucasokamura/miniconda3/envs/probing-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: MPS backend out of memory (MPS allocated: 29.96 GiB, other allocations: 160.88 MiB, max allowed: 30.19 GiB). Tried to allocate 130.82 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).